In [ ]:
import torch
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ---------------------------
# Device helpers
# ---------------------------
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"

device = get_device()

# ---------------------------
# Page config
# ---------------------------
st.set_page_config(page_title="Mistral 7B Local Chat", layout="centered")
st.title("Mistral 7B Instruct (Local)")

# ---------------------------
# Load model (cached)
# ---------------------------
@st.cache_resource
def load_model():
    model_name = "mistralai/Mistral-7B-Instruct-v0.1"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    quantization_config = None
    if device == "cuda":
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device != "cpu" else torch.float32,
        quantization_config=quantization_config,
    )

    model.to(device)
    model.eval()

    if device == "mps":
        model.config.use_cache = False

    return tokenizer, model

tokenizer, model = load_model()

# ---------------------------
# User input
# ---------------------------
prompt = st.text_area(
    "Enter your prompt:",
    placeholder="Tell me about fantasy football...",
    height=120,
)

submit = st.button("🚀 Submit")

# ---------------------------
# Inference
# ---------------------------
if submit and prompt.strip():
    with st.spinner("Generating response..."):
        formatted_prompt = f"[INST] {prompt} [/INST]"

        inputs = tokenizer(formatted_prompt, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=300,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )

        response = tokenizer.decode(
            output_ids[0],
            skip_special_tokens=True
        )

    st.subheader("🧠 Response")
    st.write(response)


Using device: mps


Loading checkpoint shards: 100%|██████████████████| 2/2 [00:31<00:00, 15.62s/it]
